In [0]:
%sh
wget https://github.com/Atulmali1055/Azure-End-to-End-Data-Engineering-Project-Data-Migration-from-Onprem-to-Cloud/blob/main/Azure-End-to-End-Data-Engineering-Project-Data-Migration-from-Onprem-to-Cloud-main/Extracted_Raw_Data_Fron_Onpremise/salesorderheader.parquet -O /Volumes/dev/bronze/managed_vol/Files/salesorderheader.parquet


In [0]:
file_Names = [f.name for f in dbutils.fs.ls("/Volumes/dev/bronze/managed_vol/Files/") if f.name.endswith(".parquet")]

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
table_Names = [s.split(".")[0] for s in file_Names]
print(table_Names)

In [0]:
display(dbutils.fs.ls("dbfs:/Volumes/dev/bronze/managed_vol/Files/"))


In [0]:
df = spark.read.option("ignoreCorruptFiles", "true").parquet("dbfs:/Volumes/dev/bronze/managed_vol/Files/customer.parquet")
df.show()

In [0]:
%sql
select * from dev.bronze.address

In [0]:
for i in range(len(table_Names)):
    df = spark.read.parquet(f"/Volumes/dev/bronze/managed_vol/Files/{file_Names[i]}")
    df = df.withColumn("ModifiedDate", date_format(from_utc_timestamp(df["ModifiedDate"].cast(TimestampType()),
"Asia/Kolkata"), "yyyy-MM-dd"))
    df = df.withColumn("TodaysDate", date_format(from_utc_timestamp(current_timestamp(), "Asia/Kolkata"), "yyyy-MM-dd"))
    df.write.mode("overwrite").saveAsTable(table_Names[i])